# FLUX Phase 3: Gravitational Relevance (GR)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Replace multi-head attention with physics-based gravitational relevance. Concepts with more accumulated evidence become "heavier" and exert stronger pull on incoming queries. O(log n) complexity via spatial index vs O(n²) for attention.
>
> **Milestone:** This phase produces **the first text output from FLUX** via a SanityDecoder wired to the end of the pipeline.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phases 1 & 2** — Verify CSE + ResonanceField checkpoints
3. **Smoke Test** — Verify GravitationalRelevance builds and runs
4. **Train** — Benchmark GR vs attention + wire up full pipeline
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — O(log n) complexity verified by latency scaling
7. **Test 2** — Retrieval precision@k (precision@1 > 0.8)
8. **Test 3** — Negative mass repulsion (10/10 concepts correctly repelled)
9. **Test 4** — End-to-end text reconstruction (15/20 recognizable)
10. **Demo 1** — Speed vs attention across sequence lengths
11. **Demo 2** — Evidence mass accumulation visualization
12. **Demo 3** — *** FIRST TEXT OUTPUT FROM FLUX *** (milestone)
13. **Interactive** — Custom mass exploration + reinforce/contradict
14. **Results** — View RESULTS_PHASE_3.md + full log
15. **Final** — Upload logs → HuggingFace & push to GitHub
16. **Artifacts** — Save checkpoint, logs, results to Kaggle output

### Key Physics:
- `mass = evidence` — concepts grow heavier with more observations
- `negative mass = contradiction` — disproven concepts repel future queries
- `F = G·m / r²` — heavier + closer = stronger gravitational pull
- O(log n) via FAISS / KDTree spatial index instead of all-pairs attention

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get the absolute
#    latest code from GitHub.
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")

    # Ensure remote points to the correct repo (may be stale from earlier clone)
    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', REPO_URL],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )

    # Discard any local modifications so pull never fails
    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)

    # Fetch + hard-reset to origin/main so we always get the absolute latest
    fetch = subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR),
                           capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f"  ⚠ git fetch failed: {fetch.stderr.strip()}")
    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())

    # Show what commit we landed on
    head = subprocess.run(
        ['git', 'log', '--oneline', '-3'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    # Remove stale directory if it exists without .git
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 2. Flush cached Python modules so the kernel
#    picks up the freshly-pulled code, not stale
#    imports from a previous cell run.
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'sanity_decoder', 'gravity', 'cse', 'field', 'flux_utils',
        'benchmark_attention', 'mass_tracker', 'spatial_index',
        'wave_types', 'interference', 'attractor', 'field_ops',
        'negative_mass', 'train_', 'demo_', 'test_',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules: {_stale[:5]}{'...' if len(_stale) > 5 else ''}")

# ─────────────────────────────────────────────
# 3. Delete stale Phase 3 checkpoint so training
#    runs fresh with the updated code.
# ─────────────────────────────────────────────
_stale_ckpt = WORK_DIR / 'checkpoints' / 'phase3.phase.pt'
if _stale_ckpt.exists():
    _stale_ckpt.unlink()
    print(f"  ✓ Deleted stale checkpoint: {_stale_ckpt.name}")

# Re-run setup.py in case directory structure changed
subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Quick sanity check: verify v2 decoder with wave_proj is present
_sd_path = WORK_DIR / 'phases' / 'phase3' / 'sanity_decoder.py'
_sd_text = _sd_path.read_text()
assert 'compute_loss' in _sd_text, "ERROR: sanity_decoder.py missing compute_loss — git pull may have failed!"
assert 'wave_proj' in _sd_text, "ERROR: sanity_decoder.py missing wave_proj — v1 decoder still present!"
assert '_lcs_length' in _sd_text, "ERROR: sanity_decoder.py missing _lcs_length — old metrics still present!"
print(f"  ✓ sanity_decoder.py verified: compute_loss + wave_proj + LCS metrics present")

print("✓ setup.py refreshed")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase3'))

# ── Force-reload all project modules (in case kernel cached old versions) ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'sanity_decoder', 'gravity', 'cse', 'field',
        'benchmark_attention', 'mass_tracker', 'spatial_index',
        'wave_types', 'interference', 'attractor', 'field_ops',
        'negative_mass',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Phase 3 Logger ──
log = PhaseLogger(phase=3)
log.separator("Phase 3: Gravitational Relevance")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# ── Load HuggingFace token ──
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token — checkpoint upload will be skipped")
    print("  ⚠ No HF token — add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phase 1 & Phase 2 Checkpoints + Smoke Test

Verifies the full checkpoint chain before Phase 3 builds on top of it.
- **Phase 1 (CSE):** Encodes text → semantic wave [seq_len, 432]
- **Phase 2 (Field):** Stores patterns as energy attractors in 3D field
- **Phase 3 (GR):** Will wire onto the field output

In [ ]:
log.cell_start("Cell 4 — Load Phase 1 & Phase 2 + Smoke Test")

import torch
import torch.nn.functional as F

# Force-reimport phase modules (ensures latest code from git pull)
import importlib
for _m in ['cse', 'field', 'gravity', 'sanity_decoder', 'mass_tracker', 'spatial_index',
           'benchmark_attention', 'negative_mass']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from cse import ContinuousSemanticEncoder
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from gravity import GravitationalRelevance
from sanity_decoder import SanityDecoder

# ── Verify SanityDecoder has the trained-decoder API ──
assert hasattr(SanityDecoder, 'compute_loss'), \
    "SanityDecoder is missing compute_loss — stale module cached! Restart kernel."
print("  ✓ SanityDecoder API verified (compute_loss present)")

# ── Load Phase 1 (CSE) ──
ckpt1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**ckpt1.get('config', {}))
cse.load_state_dict(ckpt1['state_dict'])
cse = cse.to(DEVICE).eval()
for p in cse.parameters():
    p.requires_grad = False

# Smoke test CSE
test_wave = cse.encode("smoke test Phase 1")
assert test_wave.full.shape[-1] == 432, f"Bad wave dim: {test_wave.full.shape}"
assert not torch.isnan(test_wave.full).any(), "NaN in wave!"
log.success(f"Phase 1 CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")
log.success(f"CSE smoke test: wave shape {test_wave.full.shape}")
print(f"  ✓ Phase 1 (CSE): wave shape {test_wave.full.shape}")

# ── Load Phase 2 (ResonanceField) ──
ckpt2 = load_checkpoint(2)
field_cfg = ckpt2.get('config', {}).get('field', {})
field = ResonanceField(**field_cfg)
field.load_state_dict(ckpt2['state_dict'])
field = field.to(DEVICE).eval()
for p in field.parameters():
    p.requires_grad = False

# Smoke test Field
vec = test_wave.full.mean(dim=0).to(DEVICE)
field_out, sims, locs = field.query(vec, k=4)
assert not torch.isnan(field_out).any(), "NaN in field query!"
log.success(f"Phase 2 Field loaded: {sum(p.numel() for p in field.parameters()):,} params")
log.success(f"Field smoke test: query → {field_out.shape}, top_sim={sims[0].item():.4f}")
print(f"  ✓ Phase 2 (ResonanceField): field {FIELD_H}×{FIELD_W}×{FIELD_D}×{FIELD_FEATURES}")

# ── Smoke test GravitationalRelevance ──
gr_test = GravitationalRelevance(feature_dim=512, device=DEVICE).to(DEVICE)
x_test = torch.randn(1, 16, 512, device=DEVICE)
out_test = gr_test(x_test)
assert out_test.shape == x_test.shape, f"GR shape mismatch: {out_test.shape}"
log.success(f"GravitationalRelevance smoke test: {x_test.shape} → {out_test.shape}")
print(f"  ✓ Phase 3 (GR): {x_test.shape} → {out_test.shape}")

# ── Smoke test SanityDecoder ──
dec_test = SanityDecoder(feature_dim=512, device=DEVICE).to(DEVICE)
result = dec_test.reconstruct_and_compare("hello world", out_test.squeeze(0))
log.success(f"SanityDecoder smoke test: recognizable={result['recognizable']}")
print(f"  ✓ SanityDecoder: output='{result['reconstructed'][:20]}...'")

del gr_test, dec_test, x_test, out_test
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n  Phase 1 model: https://huggingface.co/UnseenGAP/FLUX (phase1.phase.pt)")
print("  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
log.cell_end("Cell 4 — Load Phase 1 & Phase 2 + Smoke Test", "PASS")

---
## Cell 5: Training / Load from Checkpoint

> **Smart skip:** If a Phase 3 checkpoint already exists locally (`checkpoints/phase3.phase.pt`) or was previously uploaded to HuggingFace Hub, this cell loads it directly and skips all training. You can re-run the notebook from this cell onward to go straight to tests, demos, and results without re-training.

**If training from scratch — four stages:**
- **Stage A:** Run speed benchmark (GR vs PyTorch attention at 6 seq lengths)
- **Stage B1:** Warm up GR by feeding Phase 2 field output through it (builds mass tracker). Cache `(text, wave_vec, gr_output)` triples for decoder training.
- **Stage B2:** **Train SanityDecoder** — maps GR output features **+ CSE wave identity** → text bytes via cross-entropy. The wave vector is the text's own fingerprint — without it, different inputs produce identical field neighbors and the decoder can only memorize one average pattern.
- **Stage C:** Wire full pipeline — run first end-to-end pipeline check with honest metrics (LCS ratio, trigram overlap, char accuracy)

~5 min on GPU • ~30 min on CPU (skip entirely if checkpoint already saved)

In [ ]:
log.cell_start("Cell 5 — Training / Load from Checkpoint")

import time
import torch
import torch.nn.functional as F
from gravity import GravitationalRelevance
from sanity_decoder import SanityDecoder, pipeline_check
from benchmark_attention import run_benchmark

# ─────────────────────────────────────────────────────────────────
# SKIP GUARD — if Phase 3 checkpoint already exists (locally or on
# HuggingFace Hub), load it directly and skip all training stages.
# This lets you re-run the notebook from Cell 6 onward without
# re-training when a checkpoint was already saved.
# ─────────────────────────────────────────────────────────────────
_PHASE3_FRESH_TRAIN = not checkpoint_exists(3)

if not _PHASE3_FRESH_TRAIN:
    # ── FAST PATH: restore from saved checkpoint ──────────────────
    print("  ✓ Phase 3 checkpoint found — loading saved model (skipping training)")
    log.info("Phase 3 checkpoint exists — loading instead of training")

    ckpt3 = load_checkpoint(3)
    gr = GravitationalRelevance.load_state(ckpt3['phase3_gr_state'], device=DEVICE)
    gr = gr.to(DEVICE)

    decoder = SanityDecoder(feature_dim=512, device=DEVICE).to(DEVICE)
    decoder.load_state_dict(ckpt3['phase3_decoder_state'])
    decoder.eval()

    # Restore stored benchmark results and metrics so Cell 6 / demos work
    benchmark_results = ckpt3.get('benchmark_results', [])
    _stored_metrics   = ckpt3.get('metrics', {})
    elapsed           = _stored_metrics.get('training_time_seconds', 0.0)

    # Run a quick pipeline check to populate pipeline_results
    pipeline_test_inputs = [
        "the cat sat on the mat",
        "hello world",
        "artificial intelligence",
        "the quick brown fox",
        "once upon a time",
    ]
    pipeline_results = []
    with torch.no_grad():
        for text in pipeline_test_inputs:
            result = pipeline_check(cse, field, gr, decoder, text, verbose=True)
            pipeline_results.append(result)

    pass_count = sum(r['recognizable'] for r in pipeline_results)
    log.success(f"Checkpoint loaded — pipeline check: {pass_count}/{len(pipeline_test_inputs)} recognizable")
    print(f"  ✓ Checkpoint loaded  ({Path('checkpoints/phase3.phase.pt').stat().st_size/1e6:.1f} MB)")
    print(f"  ✓ Benchmark results: {len(benchmark_results)} seq lengths restored from checkpoint")
    print(f"  ✓ Pipeline check:    {pass_count}/{len(pipeline_test_inputs)} recognizable")
    print("\n  ⏩ Skipping training — continuing straight to tests, demos & results")

else:
    # ── FULL TRAINING PATH ────────────────────────────────────────
    start_time = time.time()

    # ── Build Phase 3 components ──
    gr = GravitationalRelevance(
        feature_dim=512,
        k_neighbors=32,
        base_mass=1.0,
        mass_growth_rate=0.01,
        negative_mass_rate=0.05,
        device=DEVICE,
    ).to(DEVICE)
    decoder = SanityDecoder(feature_dim=512, device=DEVICE).to(DEVICE)

    param_count_gr  = sum(p.numel() for p in gr.parameters())
    param_count_dec = sum(p.numel() for p in decoder.parameters())
    log.metric("GR parameters", f"{param_count_gr:,}")
    log.metric("Decoder parameters", f"{param_count_dec:,}")
    print(f"  GR parameters:      {param_count_gr:,}")
    print(f"  Decoder parameters: {param_count_dec:,}")

    # ────────────────────────────────────────────
    # Stage A: Speed Benchmark vs Attention
    # ────────────────────────────────────────────
    print("\n" + "="*65)
    print("  Stage A: GravitationalRelevance vs PyTorch Attention")
    print("="*65)
    benchmark_results = run_benchmark(gr, device=DEVICE)
    log.success(f"Benchmark complete: {len(benchmark_results)} seq lengths tested")

    # ────────────────────────────────────────────
    # Stage B: Warmup via Field Output
    # ────────────────────────────────────────────
    print("\n  Stage B: GR Warmup — feeding Phase 2 field output...")
    try:
        from datasets import load_dataset
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        warmup_texts = [
            item['text'].strip()[:200] for item in dataset
            if len(item['text'].strip()) > 50 and not item['text'].strip().startswith('=')
        ][:100]
        log.success(f"Loaded {len(warmup_texts)} texts from WikiText-2 for warmup")
        print(f"  ✓ Loaded {len(warmup_texts)} texts from WikiText-2")
    except Exception as e:
        log.warning(f"WikiText-2 unavailable ({e}), using built-in texts")
        warmup_texts = [
            "The cat sat on the mat and watched the birds.", "Quantum mechanics at atomic scales.",
            "Machine learning algorithms discover hidden patterns.", "The ancient civilization built pyramids.",
            "Photosynthesis converts carbon dioxide into glucose.", "Democracy requires active participation.",
            "Neural networks consist of layers of neurons.", "The orchestra performed Beethoven's symphony.",
            "Climate change challenges agricultural production.", "The economy grew in the second quarter.",
            "Stars are giant balls of plasma fusing hydrogen.", "The ocean covers 71 percent of Earth.",
            "Evolution explains the diversity of life on Earth.", "The speed of light is constant in a vacuum.",
            "Computers process binary information at high speed.",
        ] * 7

    # ── B1: Populate mass tracker + cache (text, wave_vec, gr_output) triples ──
    gr.train()
    cached_triples = []  # Store (text, wave_vec, gr_output) for decoder training
    with torch.no_grad():
        field.eval()
        cse.eval()
        warmup_sims = []
        for i, text in enumerate(warmup_texts[:80]):
            wave = cse.encode(text)
            wave_mean = wave.full.mean(dim=0)               # [432] — text identity
            vec = wave_mean.to(DEVICE)
            field_out, sims, _ = field.query(vec, k=8)
            gr_out = gr(field_out.unsqueeze(0)).squeeze(0)
            warmup_sims.append(sims[0].item())
            # Cache triple: text, wave (identity signal), GR features (field context)
            cached_triples.append((text, wave_mean.detach().clone(), gr_out.detach().clone()))
            if (i + 1) % 20 == 0:
                avg_sim = sum(warmup_sims[-20:]) / 20
                msg = f"Warmup {i+1:3d}/80: avg_top_sim={avg_sim:.4f}, mass_count={gr.mass_tracker.count}"
                log.info(msg)
                print(f"    {msg}")

    log.success(f"GR warmup: {gr.mass_tracker.count} concepts accumulated in mass tracker")
    print(f"  ✓ Mass tracker: {gr.mass_tracker.count} concepts, "
          f"avg_mass={gr.mass_tracker.masses[:gr.mass_tracker.count].mean().item():.4f}")

    # ────────────────────────────────────────────
    # Stage B2: Train SanityDecoder (wave-aware)
    # ────────────────────────────────────────────
    # The decoder now receives BOTH:
    #   - The CSE wave vector (text-specific identity signal)
    #   - The GR field features (contextual knowledge)
    # This way different inputs produce different outputs,
    # because the wave fingerprint distinguishes them.
    # ────────────────────────────────────────────
    print("\n" + "="*65)
    print("  Stage B2: Training SanityDecoder (wave + GR features → text)")
    print("="*65)

    decoder.train()
    dec_optimizer = torch.optim.AdamW(decoder.parameters(), lr=5e-4, weight_decay=1e-4)
    dec_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(dec_optimizer, T_max=80, eta_min=1e-5)
    NUM_DECODER_EPOCHS = 80
    best_loss = float('inf')
    patience, patience_limit = 0, 10
    import random

    for epoch in range(NUM_DECODER_EPOCHS):
        epoch_loss = 0.0
        # Shuffle triples each epoch
        indices = list(range(len(cached_triples)))
        random.shuffle(indices)

        for idx in indices:
            text, wave_vec, gr_features = cached_triples[idx]
            gr_features = gr_features.to(DEVICE)
            wave_vec_d  = wave_vec.to(DEVICE)
            loss = decoder.compute_loss(gr_features, text, wave_vec=wave_vec_d)
            dec_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            dec_optimizer.step()
            epoch_loss += loss.item()

        dec_scheduler.step()
        avg_loss = epoch_loss / len(cached_triples)
        current_lr = dec_optimizer.param_groups[0]['lr']

        # Check a sample reconstruction every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            decoder.eval()
            with torch.no_grad():
                sample_text, sample_wave, sample_feat = cached_triples[0]
                sample_out = decoder.decode(sample_feat.to(DEVICE), wave_vec=sample_wave.to(DEVICE))
            decoder.train()
            msg = f"Epoch {epoch+1:3d}/{NUM_DECODER_EPOCHS}: loss={avg_loss:.4f}, lr={current_lr:.1e}, sample='{sample_out[:50]}'"
            log.info(msg)
            print(f"    {msg}")

        # Early stopping (only after epoch 30 with generous patience)
        if avg_loss < best_loss - 0.005:
            best_loss = avg_loss
            patience = 0
        else:
            patience += 1
        if patience >= patience_limit and epoch >= 30:
            log.info(f"Early stopping at epoch {epoch+1} (loss plateaued at {avg_loss:.4f})")
            print(f"    Early stopping at epoch {epoch+1}")
            break

    decoder.eval()
    log.success(f"Decoder trained: final loss={avg_loss:.4f}")
    print(f"  ✓ Decoder training complete: final loss={avg_loss:.4f}")

    # ────────────────────────────────────────────
    # Stage C: Pipeline Check (FIRST TEXT OUTPUT)
    # Honest metrics: LCS ratio, trigram overlap, char accuracy
    # ────────────────────────────────────────────
    print("\n" + "="*65)
    print("  Stage C: First End-to-End Pipeline Check (honest metrics)")
    print("="*65)

    pipeline_test_inputs = [
        "the cat sat on the mat",
        "hello world",
        "artificial intelligence",
        "the quick brown fox",
        "once upon a time",
    ]
    pipeline_results = []
    for text in pipeline_test_inputs:
        result = pipeline_check(cse, field, gr, decoder, text, verbose=True)
        pipeline_results.append(result)
        log.metric(f"pipeline_check('{text[:20]}')",
                   f"lcs={result['lcs_ratio']:.2f} tri={result['trigram_overlap']:.2f} rec={result['recognizable']}")

    pass_count = sum(r['recognizable'] for r in pipeline_results)
    avg_lcs = sum(r['lcs_ratio'] for r in pipeline_results) / len(pipeline_results)
    avg_tri = sum(r['trigram_overlap'] for r in pipeline_results) / len(pipeline_results)
    print(f"\n  Pipeline check: {pass_count}/{len(pipeline_test_inputs)} recognizable")
    print(f"  Avg LCS ratio:     {avg_lcs:.2%}")
    print(f"  Avg trigram overlap: {avg_tri:.2%}")
    log.success(f"Pipeline: {pass_count}/{len(pipeline_test_inputs)} recognizable, avg_lcs={avg_lcs:.2%}")

    elapsed = time.time() - start_time
    log.metric("training_time", f"{elapsed:.1f}s")
    print(f"\n  ✓ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")

log.cell_end("Cell 5 — Training / Load from Checkpoint", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

> **Smart skip:** If the checkpoint was already saved (Cell 5 loaded it rather than training), this cell skips the save and shows the existing checkpoint path. Upload is also skipped so the already-published model on HuggingFace is not overwritten.

If freshly trained, saves to `checkpoints/phase3.phase.pt` and uploads to `https://huggingface.co/UnseenGAP/FLUX`.

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

from datetime import datetime

if not _PHASE3_FRESH_TRAIN:
    # ── Checkpoint already existed — nothing to save ──────────────
    ckpt_path = Path('checkpoints/phase3.phase.pt')
    size_mb   = ckpt_path.stat().st_size / 1e6 if ckpt_path.exists() else 0
    print(f"  ⏩ Checkpoint already saved — skipping save step")
    print(f"     Local:  {ckpt_path} ({size_mb:.1f} MB)")
    print(f"     Remote: https://huggingface.co/UnseenGAP/FLUX")
    log.info("Checkpoint already existed — save step skipped")
else:
    # ── Freshly trained — save and upload ────────────────────────
    pass_rate  = sum(r['recognizable'] for r in pipeline_results) / len(pipeline_results)
    mass_stats = gr.mass_tracker.stats()

    checkpoint_state = {
        'phase': 3,
        'timestamp': datetime.now().isoformat(),
        # Chain: include references to previous phases by config
        'phase1_config': ckpt1.get('config', {}),
        'phase2_config': ckpt2.get('config', {}),
        # Phase 3 components
        'phase3_gr_state': gr.save_state(),
        'phase3_decoder_state': decoder.state_dict(),
        # Benchmark results recorded at checkpoint time
        'benchmark_results': benchmark_results,
        # Metrics
        'metrics': {
            'pipeline_check_pass_rate': pass_rate,
            'mass_tracker_count': gr.mass_tracker.count,
            'mass_tracker_mean': mass_stats.get('mean_mass', 0.0),
            'training_time_seconds': elapsed,
        },
    }
    save_checkpoint(3, checkpoint_state)

    ckpt_path = Path('checkpoints/phase3.phase.pt')
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
        print(f"  ✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    else:
        log.error("Checkpoint NOT found — save may have failed")

    # ── Upload to HuggingFace Hub ──
    uploaded = upload_checkpoint_to_hf(phase=3, hf_token=HF_TOKEN)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
        print("  ✓ Uploaded to HuggingFace Hub")
    else:
        log.warning("Checkpoint upload skipped — check HF_TOKEN")
        print("  ⚠ Upload skipped — no HF token")

    # Upload intermediate log
    upload_logs_to_hf(phase=3, hf_token=HF_TOKEN)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")


---
## Cells 7–10: Tests

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| 1 | O(log n) complexity | Sub-linear scaling R² > 0.85; avg doubling growth < 3.0x |
| 2 | Retrieval precision@k | Precision@1 > 0.8, Precision@10 > 0.7 |
| 3 | Negative mass repulsion | 10/10 contradicted concepts repel correctly |
| 4 | End-to-end text reconstruction (honest) | ≥10/20 recognizable (LCS > 15%) + avg LCS > 10% |

In [ ]:
log.cell_start("Cell 7 — Test 1: O(log n) Complexity")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run test_phase3_test1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 7 — Test 1: O(log n) Complexity", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: Retrieval Precision@k")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run test_phase3_test2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 8 — Test 2: Retrieval Precision@k", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Negative Mass Repulsion")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run test_phase3_test3.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 9 — Test 3: Negative Mass Repulsion", "PASS")

In [ ]:
log.cell_start("Cell 10 — Test 4: End-to-End Text Reconstruction")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run test_phase3_test4.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 10 — Test 4: End-to-End Text Reconstruction", "PASS")

---
## Cells 11–13: Demos

| Demo | Description |
|------|-------------|
| 1 | GR vs attention speed table + matplotlib chart |
| 2 | Evidence mass accumulation — feed astronomy texts, show mass growth + contradicted concept |
| 3 | **★ FIRST TEXT OUTPUT** — Full CSE→Field→GR→Decoder pipeline end-to-end |

In [ ]:
log.cell_start("Cell 11 — Demo 1: GR Speed vs Attention")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run demo_phase3_demo1.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase3' / 'demo3_speed_comparison.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("Speed comparison chart generated")
else:
    log.warning("Speed chart not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 11 — Demo 1: GR Speed vs Attention", "PASS")

In [ ]:
log.cell_start("Cell 12 — Demo 2: Mass Accumulation Visualization")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run demo_phase3_demo2.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase3' / 'demo3_mass_distribution.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("Mass distribution chart generated")
else:
    log.warning("Mass chart not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 12 — Demo 2: Mass Accumulation Visualization", "PASS")

In [ ]:
log.cell_start("Cell 13 — Demo 3: ★ FIRST TEXT OUTPUT FROM FLUX ★")

os.chdir(str(WORK_DIR / 'phases' / 'phase3'))
%run demo_phase3_demo3.py
os.chdir(str(WORK_DIR))

log.success("MILESTONE: First text output demonstrated via full pipeline")
log.cell_end("Cell 13 — Demo 3: First Text Output", "PASS")

---
## Cell 14: Interactive Exploration
Explore the mass tracker, reinforce/contradict concepts, and run custom pipeline checks.

In [ ]:
log.cell_start("Cell 14 — Interactive Exploration")

import torch
import torch.nn.functional as F
from gravity import GravitationalRelevance
from sanity_decoder import SanityDecoder, pipeline_check

# ── Reload GR from the saved checkpoint ──
ckpt3 = load_checkpoint(3)
gr_interactive = GravitationalRelevance.load_state(ckpt3['phase3_gr_state'], device='cpu')
dec_interactive = SanityDecoder(feature_dim=512, device='cpu')
dec_interactive.load_state_dict(ckpt3['phase3_decoder_state'])
dec_interactive.eval()

print("  Mass Tracker Statistics:")
print("  " + "-" * 45)
stats = gr_interactive.mass_tracker.stats()
for k, v in stats.items():
    if isinstance(v, float):
        print(f"    {k:<25}: {v:.4f}")
    else:
        print(f"    {k:<25}: {v}")
    log.metric(k, str(v))

# ── Custom pipeline checks ──
print("\n  Custom Pipeline Checks (honest metrics):")
print("  " + "-" * 72)
print(f"  {'Input Text':<30} {'Recognize':>9} {'LCS':>6} {'Trigram':>8} {'CharAcc':>8}")
print("  " + "-" * 72)

test_pairs = [
    "the cat sat on the mat",
    "hello world",
    "deep learning",
    "the sun rises in the east",
    "gravitational relevance",
    "FLUX architecture",
]

for text in test_pairs:
    result = pipeline_check(
        cse.to('cpu'), field.to('cpu'), gr_interactive, dec_interactive, text, verbose=False
    )
    icon = "✓" if result['recognizable'] else "✗"
    print(f"  {icon} {text:<29} {str(result['recognizable']):>9} "
          f"{result['lcs_ratio']:>5.2f} {result['trigram_overlap']:>7.2f} "
          f"{result['char_accuracy']:>7.2f}")
    log.metric(f"explore('{text[:20]}')",
               f"lcs={result['lcs_ratio']:.2f} tri={result['trigram_overlap']:.2f}")

# ── Reinforce / contradict demo ──
print("\n  Reinforce / Contradict Concepts:")
print("  " + "-" * 45)
# CSE wave is 432-dim; mass_tracker stores 512-dim features → project first
test_wave = cse.encode("Pluto is a planet").full.mean(dim=0).to('cpu')
with torch.no_grad():
    test_concept = field.to('cpu').wave_to_feature(test_wave)  # [432] → [512]

before_stats = gr_interactive.mass_tracker.stats()
gr_interactive.reinforce(test_concept, strength=5.0)
after_reinforce = gr_interactive.mass_tracker.stats()
gr_interactive.contradict(test_concept, strength=20.0)
after_contradict = gr_interactive.mass_tracker.stats()

print(f"  Before reinforce:     mean_mass = {before_stats.get('mean_mass', 0):.4f}")
print(f"  After reinforce ×5:   mean_mass = {after_reinforce.get('mean_mass', 0):.4f}")
print(f"  After contradict ×20: mean_mass = {after_contradict.get('mean_mass', 0):.4f}")
print(f"  Negative concepts:    {after_contradict.get('negative_count', 0)}")

print("\n  ✓ Interactive exploration complete")

# Restore to GPU for consistency
cse = cse.to(DEVICE)
field = field.to(DEVICE)
del gr_interactive, dec_interactive

log.cell_end("Cell 14 — Interactive Exploration", "PASS")

---
## Cell 15: View Results Summary & Full Log

In [ ]:
log.cell_start("Cell 15 — Results Summary & Log")

# ── Show RESULTS_PHASE_3.md if it exists ──
from IPython.display import Markdown, display

results_path = WORK_DIR / 'phases' / 'phase3' / 'RESULTS_PHASE_3.md'
if results_path.exists():
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    print("  ⚠ RESULTS_PHASE_3.md not yet generated — run tests first")
    print("  Tests auto-generate this file via PhaseResults utility.")
    log.warning("No RESULTS_PHASE_3.md found")

# ── Show full log ──
print("\n" + "=" * 60)
print("FULL PHASE 3 LOG")
print("=" * 60)
print(log.get_log_contents())

log.cell_end("Cell 15 — Results Summary & Log", "PASS")

---
## Cell 16: Final Upload — Logs to HuggingFace & GitHub
Pushes the complete Phase 3 log, results, and checkpoint to both platforms.

In [ ]:
log.cell_start("Cell 16 — Final Upload")

# ── Upload final log to HuggingFace ──
upload_logs_to_hf(phase=3, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

# ── Commit logs + results to GitHub ──
git_commit_and_push(
    files=[
        'logs/phase3.log',
        'results/',
        'phases/phase3/RESULTS_PHASE_3.md',
    ],
    message='Phase 3: Gravitational Relevance — training complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 16 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 3 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 4 — Thermodynamic Learning")
print("=" * 60)

---
## Cell 17: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 17 — Save Kaggle Artifacts")

import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# ── Checkpoints ──
for fname in ['phase3.phase.pt']:
    src = WORK_DIR / 'checkpoints' / fname
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Checkpoint: {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# ── Results and logs ──
for fpath in [
    WORK_DIR / 'phases' / 'phase3' / 'RESULTS_PHASE_3.md',
    WORK_DIR / 'logs' / 'phase3.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

# ── Demo images ──
for img in (WORK_DIR / 'phases' / 'phase3').glob('*.png'):
    shutil.copy2(str(img), str(output_dir / img.name))
    print(f"  ✓ {img.name}")

print(f"\n  Kaggle output artifacts:")
for f in sorted(output_dir.iterdir()):
    print(f"    {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

log.cell_end("Cell 17 — Save Kaggle Artifacts", "PASS")

print("\n" + "=" * 60)
print("  PHASE 3: Gravitational Relevance — ALL DONE ✓")
print("=" * 60)

---
## Phase 3 Acceptance Criteria

| Criteria | Cell | Target |
|---|---|---|
| GR scales sub-linearly (growth < 3x per doubling) | Test 1 / Demo 1 | scaling verified |
| O(log n) complexity verified | Test 1 | Log fit R² > 0.9 |
| Retrieval precision@10 > 0.7 | Test 2 | ✓ |
| Retrieval precision@1 > 0.8 | Test 2 | ✓ |
| Negative mass repulsion correct | Test 3 | 10/10 concepts |
| End-to-end text reconstruction (honest metrics) | Test 4 | ≥ 10/20 recognizable (LCS > 15%) |
| Pipeline check runs without error | Demo 3 | ✓ |
| SanityDecoder produces input-specific output | Demo 3 | Different inputs → different outputs |
| Wave identity signal reaches decoder | Cell 5 | wave_proj present in decoder |
| Phase 1 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 2 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 3 checkpoint saved | Cell 6 | `checkpoints/phase3.phase.pt` |
| Checkpoint uploaded to HuggingFace | Cell 6 | `UnseenGAP/FLUX` |
| Logs written to logs/phase3.log | All cells | ✓ |
| RESULTS_PHASE_3.md generated | Tests | ✓ |

### Honest Assessment Notes
- GR is **not** faster than fused CUDA attention in absolute time (Python/CPU spatial index overhead)
- GR **does** scale sub-linearly (O(n log n) growth rate verified)
- "Recognizable" now requires LCS ratio > 15% or char accuracy > 8% — not just common letters appearing
- SanityDecoder receives both wave identity (text fingerprint) + field context (knowledge)